# 00 — CRISP-DM Methodology Reconstruction Suite
## Decision Dashboard with Explainable AI and Human Override for Two-Stage Screening of Unrecognized HbA1c-Defined Dysglycemia

**Author:** Felix (Thesis Research)
**Data:** CDC / NCHS NHANES August 2021 – August 2023
**Framework:** CRISP-DM (Cross-Industry Standard Process for Data Mining)

---

## Executive Summary

This notebook suite reconstructs and documents the **complete research methodology** for a non-laboratory screening system detecting unrecognized dysglycemia. The suite is organized following the **CRISP-DM** framework and serves as an auditable chain of evidence connecting raw survey microdata to the deployed clinical decision-support dashboard.

**Key Research Artifacts:**
- **Primary Model:** Generalized Additive Model (GAM) with 7 non-laboratory predictors
- **Frozen Decision Threshold:** τ = 0.1389 (pre-specified on development data)
- **Final Test Performance:** Sensitivity 86.39%, Specificity 42.51%, ROC-AUC 0.7277 (N=812)
- **Explainability:** GAM-native additive decomposition (zero SHAP dependence)
- **User Evaluation:** Protocol E1 v1.0.3 (Locked, awaiting institutional clearance)

---

## CRISP-DM Navigation Map

| # | Notebook | CRISP-DM Phase | Key Deliverable |
|:--|:---------|:---------------|:----------------|
| **01** | `01_BUSINESS_UNDERSTANDING.ipynb` | Business Understanding | Problem framing, task definition, scope boundaries |
| **02** | `02_DATA_UNDERSTANDING.ipynb` | Data Understanding | Raw NHANES microdata inventory, SEQN audit, quality check |
| **03** | `03_DATA_PREPARATION.ipynb` | Data Preparation | Cohort E construction, 7 predictors, preprocessing pipeline |
| **04** | `04_MODELING.ipynb` | Modeling | 5-fold CV, LR vs GAM vs DLNN, OOF model selection |
| **05** | `05_THRESHOLD_SELECTION.ipynb` | Modeling (Decision Calibration) | Constrained optimization → τ = 0.1389 |
| **06** | `06_FINAL_EVALUATION.ipynb` | Evaluation | Confirmatory held-out test (N=812), bootstrap CI |
| **07** | `07_XAI_HUMAN_REVIEW.ipynb` | Deployment | GAM term decomposition, fidelity proof, override semantics |
| **08** | `08_STAGE2_USER_EVALUATION.ipynb` | Evaluation & Governance | Stage-2 HbA1c protocol, E1 user study methodology |

---

## End-to-End Dataflow

```
Raw NHANES XPT (8 files, N=11,933)
    │
    ▼
Cohort E Filtering (Age ≥18, No known DM/PreDM, Valid HbA1c)
    │
    ▼
Canonical Expanded Dataset (N=4,044; 3,106 normal, 938 dysglycemia)
    │
    ├── 80% Development (N=3,232) ──→ 5-Fold Stratified CV ──→ OOF Predictions
    │                                      │
    │                                      ▼
    │                              Model Selection (GAM wins on PR-AUC)
    │                                      │
    │                                      ▼
    │                              Threshold τ=0.1389 (≥90% sensitivity floor)
    │                                      │
    │                                      ▼
    │                              SPECIFICATION LOCK (Phase 4.2)
    │
    └── 20% Final Test (N=812) ──→ ONE-TIME Confirmatory Evaluation
                                           │
                                           ▼
                                   Sens=86.39%, Spec=42.51%, ROC-AUC=0.7277
                                           │
                                           ▼
                                   Dashboard Deployment (GAM + XAI + Human Override)
                                           │
                                           ▼
                                   Protocol E1: User Evaluation (LOCKED v1.0.3)
```

---

## Environment & Reproducibility

**Critical:** All computational results in this suite are derived from frozen artifacts. The notebooks read existing parquet datasets, CSV prediction files, and locked model specifications — they do NOT re-train models or re-optimize thresholds.

| Component | Specification |
|:----------|:-------------|
| Python | 3.10.11 |
| Random Seed | 42 (all splitting, CV, bootstrapping) |
| Primary Model | `pygam.LogisticGAM` (n_splines=10, λ=10.0) |
| Preprocessing | `sklearn.preprocessing.StandardScaler` (continuous only) |
| Operating Threshold | 0.1389 (immutable) |

---

## Source-of-Truth Priority

1. **Frozen Research Artifacts** (parquet, CSV, locked markdown specifications)
2. **Implementation Scripts** (`scripts_phase3/`, `scripts_phase4/`, `scripts_phase5/`)
3. **These Notebooks** (documentation layer — do NOT override source artifacts)

> **Governance Rule:** If any computed value in these notebooks disagrees with a frozen artifact, the notebook flags `SOURCE VERIFICATION REQUIRED` rather than overriding the artifact value.


In [1]:
# Environment verification
import sys
import importlib

required = ['numpy', 'pandas', 'matplotlib', 'sklearn', 'pygam']
print("=" * 60)
print("CRISP-DM NOTEBOOK SUITE — ENVIRONMENT CHECK")
print("=" * 60)
print(f"Python: {sys.version}")
for pkg in required:
    try:
        mod = importlib.import_module(pkg)
        ver = getattr(mod, '__version__', 'installed')
        print(f"  [OK] {pkg}: {ver}")
    except ImportError:
        print(f"  [MISSING] {pkg}: NOT INSTALLED")
print("=" * 60)


CRISP-DM NOTEBOOK SUITE — ENVIRONMENT CHECK
Python: 3.10.11 (tags/v3.10.11:7d4cc5a, Apr  5 2023, 00:38:17) [MSC v.1929 64 bit (AMD64)]
  [OK] numpy: 2.2.6
  [OK] pandas: 2.3.3
  [OK] matplotlib: 3.10.9
  [OK] sklearn: 1.7.2
  [OK] pygam: 0.12.0
